# GraphRAG + Qdrant PoC — Uber-like Ride Marketplace

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│  Source Data (JSON / Relational DB records)                         │
│  Drivers · Riders · Trips · Reviews · Zones                        │
└────────────────────────────┬────────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────────┐
│  GraphRAG Pipeline (NetworkX)                                       │
│  Entity extraction → Relationship mapping → Community summaries     │
└──────────┬──────────────────────────────────────┬───────────────────┘
           │                                      │
           ▼                                      ▼
    Dense Embeddings                       Sparse Vectors
    (OpenAI text-embedding-3-small)        (BM25 term weights)
           │                                      │
           └──────────────┬───────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│  Qdrant Serverless (CRUD + Hybrid Search)                           │
│  Named vectors: 'dense' + 'sparse'                                  │
│  Payload: entity metadata, graph community, source record           │
└─────────────────────────────────────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────────┐
│  Hybrid Search + GraphRAG Answer Generation                         │
│  RRF fusion → context injection → OpenAI GPT-4o response           │
└─────────────────────────────────────────────────────────────────────┘
```

### What this notebook demonstrates
1. **Data ingestion** — JSON arrays mimicking relational DB tables (drivers, riders, trips, reviews, zones)
2. **GraphRAG** — build a knowledge graph, extract communities, generate rich text summaries per entity
3. **Qdrant CRUD** — programmatic Create / Read / Update / Delete on Qdrant Serverless
4. **Hybrid search** — combine dense (semantic) + sparse (keyword/BM25) vectors with RRF fusion
5. **RAG generation** — use retrieved graph context to answer natural-language questions

## 1. Install Dependencies

In [ ]:
%pip install -q openai "qdrant-client>=1.10.0" networkx numpy pandas matplotlib python-dotenv rank-bm25 tqdm

## 2. Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load from .env file if present; otherwise set directly below
load_dotenv()

OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY",  "sk-YOUR_OPENAI_KEY")
QDRANT_URL      = os.getenv("QDRANT_URL",      "https://YOUR_CLUSTER.qdrant.io")
QDRANT_API_KEY  = os.getenv("QDRANT_API_KEY",  "YOUR_QDRANT_API_KEY")

COLLECTION_NAME = "uber_marketplace"
EMBED_MODEL     = "text-embedding-3-small"
EMBED_DIM       = 1536
CHAT_MODEL      = "gpt-4o"

print("Config loaded.")
print(f"  Qdrant URL : {QDRANT_URL}")
print(f"  Collection : {COLLECTION_NAME}")
print(f"  Embed model: {EMBED_MODEL} ({EMBED_DIM}d)")

## 3. Initialise Clients

In [ ]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector, NamedVector, NamedSparseVector,
    Filter, FieldCondition, MatchValue, MatchAny,
    SearchRequest, FusionQuery, Prefetch, Query,
    UpdateStatus, Record
)

openai_client = OpenAI(api_key=OPENAI_API_KEY)
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Verify Qdrant connectivity
info = qdrant_client.get_collections()
print(f"Connected to Qdrant. Existing collections: {[c.name for c in info.collections]}")

## 4. Source Data — Uber-like Marketplace Records

These JSON arrays simulate rows from five relational database tables:
- `drivers` — driver profiles with ratings and vehicle info
- `riders` — rider profiles
- `trips` — completed ride records
- `reviews` — post-trip reviews (bidirectional)
- `zones` — city zone definitions

In [ ]:
import json, pandas as pd

# ── Drivers ──────────────────────────────────────────────────────────────────
drivers = [
    {"driver_id": "D001", "name": "Carlos Rivera",  "city": "San Francisco", "rating": 4.92,
     "trips_completed": 2341, "vehicle": "Tesla Model 3",   "vehicle_type": "electric",
     "badge": "top_rated",  "languages": ["English", "Spanish"], "zone": "Z001"},
    {"driver_id": "D002", "name": "Priya Nair",     "city": "San Francisco", "rating": 4.87,
     "trips_completed": 1876, "vehicle": "Honda Civic",     "vehicle_type": "economy",
     "badge": "veteran",    "languages": ["English", "Hindi"],   "zone": "Z002"},
    {"driver_id": "D003", "name": "Marcus Johnson", "city": "Oakland",       "rating": 4.78,
     "trips_completed": 987,  "vehicle": "Toyota Camry",    "vehicle_type": "comfort",
     "badge": None,         "languages": ["English"],             "zone": "Z003"},
    {"driver_id": "D004", "name": "Aisha Okonkwo", "city": "San Jose",       "rating": 4.95,
     "trips_completed": 3102, "vehicle": "BMW 5 Series",    "vehicle_type": "luxury",
     "badge": "top_rated",  "languages": ["English", "Yoruba"],   "zone": "Z004"},
    {"driver_id": "D005", "name": "Wei Zhang",     "city": "San Francisco", "rating": 4.81,
     "trips_completed": 1423, "vehicle": "Prius Prime",     "vehicle_type": "electric",
     "badge": None,         "languages": ["English", "Mandarin"], "zone": "Z001"},
]

# ── Riders ───────────────────────────────────────────────────────────────────
riders = [
    {"rider_id": "R001", "name": "Jordan Lee",    "city": "San Francisco", "rating": 4.80,
     "total_trips": 312, "preferred_type": "electric", "frequent_zone": "Z001"},
    {"rider_id": "R002", "name": "Samantha Cruz", "city": "Oakland",       "rating": 4.65,
     "total_trips": 87,  "preferred_type": "economy",  "frequent_zone": "Z002"},
    {"rider_id": "R003", "name": "Ethan Park",    "city": "San Jose",       "rating": 4.95,
     "total_trips": 521, "preferred_type": "luxury",   "frequent_zone": "Z004"},
    {"rider_id": "R004", "name": "Nina Patel",    "city": "San Francisco", "rating": 4.72,
     "total_trips": 154, "preferred_type": "comfort",  "frequent_zone": "Z003"},
    {"rider_id": "R005", "name": "Omar Hassan",   "city": "Oakland",       "rating": 4.88,
     "total_trips": 229, "preferred_type": "economy",  "frequent_zone": "Z002"},
]

# ── Zones ────────────────────────────────────────────────────────────────────
zones = [
    {"zone_id": "Z001", "name": "Downtown SF",        "city": "San Francisco",
     "surge_active": True,  "avg_wait_min": 3.2, "demand_level": "high"},
    {"zone_id": "Z002", "name": "Mission District",   "city": "San Francisco",
     "surge_active": False, "avg_wait_min": 5.1, "demand_level": "medium"},
    {"zone_id": "Z003", "name": "Oakland Downtown",   "city": "Oakland",
     "surge_active": False, "avg_wait_min": 6.8, "demand_level": "low"},
    {"zone_id": "Z004", "name": "Silicon Valley Hub", "city": "San Jose",
     "surge_active": True,  "avg_wait_min": 4.0, "demand_level": "high"},
]

# ── Trips ────────────────────────────────────────────────────────────────────
trips = [
    {"trip_id": "T001", "driver_id": "D001", "rider_id": "R001",
     "pickup_zone": "Z001", "dropoff_zone": "Z002", "fare": 18.50,
     "distance_km": 5.2, "duration_min": 14, "vehicle_type": "electric",
     "status": "completed", "timestamp": "2024-03-15T08:22:00"},
    {"trip_id": "T002", "driver_id": "D004", "rider_id": "R003",
     "pickup_zone": "Z004", "dropoff_zone": "Z004", "fare": 45.00,
     "distance_km": 12.1, "duration_min": 28, "vehicle_type": "luxury",
     "status": "completed", "timestamp": "2024-03-15T09:10:00"},
    {"trip_id": "T003", "driver_id": "D002", "rider_id": "R002",
     "pickup_zone": "Z002", "dropoff_zone": "Z003", "fare": 22.00,
     "distance_km": 8.7, "duration_min": 21, "vehicle_type": "economy",
     "status": "completed", "timestamp": "2024-03-15T10:05:00"},
    {"trip_id": "T004", "driver_id": "D001", "rider_id": "R004",
     "pickup_zone": "Z001", "dropoff_zone": "Z003", "fare": 31.75,
     "distance_km": 10.3, "duration_min": 34, "vehicle_type": "electric",
     "status": "completed", "timestamp": "2024-03-15T11:30:00"},
    {"trip_id": "T005", "driver_id": "D005", "rider_id": "R001",
     "pickup_zone": "Z001", "dropoff_zone": "Z004", "fare": 38.20,
     "distance_km": 15.6, "duration_min": 42, "vehicle_type": "electric",
     "status": "completed", "timestamp": "2024-03-15T13:00:00"},
    {"trip_id": "T006", "driver_id": "D003", "rider_id": "R005",
     "pickup_zone": "Z003", "dropoff_zone": "Z002", "fare": 19.00,
     "distance_km": 7.1, "duration_min": 18, "vehicle_type": "comfort",
     "status": "completed", "timestamp": "2024-03-15T14:45:00"},
]

# ── Reviews ──────────────────────────────────────────────────────────────────
reviews = [
    {"review_id": "REV001", "trip_id": "T001", "reviewer_type": "rider",
     "reviewer_id": "R001", "reviewee_id": "D001", "rating": 5,
     "text": "Carlos was fantastic! Super smooth Tesla ride, arrived in 2 minutes, very professional."},
    {"review_id": "REV002", "trip_id": "T001", "reviewer_type": "driver",
     "reviewer_id": "D001", "reviewee_id": "R001", "rating": 5,
     "text": "Great rider, ready on time, respectful. Would love to drive Jordan again."},
    {"review_id": "REV003", "trip_id": "T002", "reviewer_type": "rider",
     "reviewer_id": "R003", "reviewee_id": "D004", "rating": 5,
     "text": "Aisha provided an exceptional luxury experience. Extremely punctual, BMW spotless."},
    {"review_id": "REV004", "trip_id": "T003", "reviewer_type": "rider",
     "reviewer_id": "R002", "reviewee_id": "D002", "rating": 4,
     "text": "Priya was friendly and got me there safely. Slightly longer route but good conversation."},
    {"review_id": "REV005", "trip_id": "T004", "reviewer_type": "rider",
     "reviewer_id": "R004", "reviewee_id": "D001", "rating": 5,
     "text": "Second time with Carlos — always reliable. The Tesla is so quiet and comfortable."},
    {"review_id": "REV006", "trip_id": "T005", "reviewer_type": "rider",
     "reviewer_id": "R001", "reviewee_id": "D005", "rating": 4,
     "text": "Wei was knowledgeable about traffic. Long trip but very comfortable."},
    {"review_id": "REV007", "trip_id": "T006", "reviewer_type": "rider",
     "reviewer_id": "R005", "reviewee_id": "D003", "rating": 4,
     "text": "Marcus was polite and the Camry was clean. Took a slightly roundabout route."},
]

print(f"Loaded: {len(drivers)} drivers, {len(riders)} riders, {len(trips)} trips, "
      f"{len(reviews)} reviews, {len(zones)} zones")

## 5. Build Knowledge Graph with NetworkX (GraphRAG)

We model the marketplace as a directed graph:
- **Node types**: Driver, Rider, Trip, Zone
- **Edge types**: COMPLETED (driver→trip), TOOK (rider→trip), PICKED_UP_IN (trip→zone), DROPPED_OFF_IN (trip→zone), REVIEWED (rider→driver)

This mirrors what Microsoft GraphRAG does — entities become nodes, relationships become edges, and communities become clusters for summarisation.

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Add driver nodes
for d in drivers:
    G.add_node(d["driver_id"], **d, node_type="driver")

# Add rider nodes
for r in riders:
    G.add_node(r["rider_id"], **r, node_type="rider")

# Add zone nodes
for z in zones:
    G.add_node(z["zone_id"], **z, node_type="zone")

# Add trip nodes + edges
for t in trips:
    G.add_node(t["trip_id"], **t, node_type="trip")
    G.add_edge(t["driver_id"], t["trip_id"],  relation="COMPLETED")
    G.add_edge(t["rider_id"],  t["trip_id"],  relation="TOOK")
    G.add_edge(t["trip_id"],   t["pickup_zone"],  relation="PICKED_UP_IN")
    G.add_edge(t["trip_id"],   t["dropoff_zone"], relation="DROPPED_OFF_IN")

# Add review edges (rider→driver)
for rev in reviews:
    if rev["reviewer_type"] == "rider":
        G.add_edge(
            rev["reviewer_id"], rev["reviewee_id"],
            relation="REVIEWED",
            rating=rev["rating"],
            text=rev["text"],
            review_id=rev["review_id"]
        )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Node type distribution
from collections import Counter
type_counts = Counter(data["node_type"] for _, data in G.nodes(data=True))
for k, v in type_counts.items():
    print(f"  {k:8s}: {v} nodes")

### 5.1 Visualise the Graph

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

colour_map = {"driver": "#4A90D9", "rider": "#7ED321", "trip": "#F5A623", "zone": "#D0021B"}
node_colours = [colour_map[G.nodes[n]["node_type"]] for n in G.nodes()]
node_sizes   = [800 if G.nodes[n]["node_type"] in ("driver", "rider") else 400 for n in G.nodes()]

pos = nx.spring_layout(G, seed=42, k=0.6)

fig, ax = plt.subplots(figsize=(14, 9))
nx.draw_networkx_nodes(G,  pos, node_color=node_colours, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G,  pos, arrows=True, arrowsize=12, edge_color="#888",
                       connectionstyle="arc3,rad=0.1", ax=ax, width=0.8)

legend = [mpatches.Patch(color=c, label=t) for t, c in colour_map.items()]
ax.legend(handles=legend, loc="upper left", fontsize=9)
ax.set_title("Uber-like Marketplace Knowledge Graph", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("knowledge_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → knowledge_graph.png")

### 5.2 Detect Communities (GraphRAG-style clustering)

In [ ]:
# Convert to undirected for community detection
UG = G.to_undirected()

from networkx.algorithms.community import greedy_modularity_communities
communities = list(greedy_modularity_communities(UG))

node_to_community = {}
for i, community in enumerate(communities):
    for node in community:
        node_to_community[node] = i

print(f"Detected {len(communities)} communities:")
for i, comm in enumerate(communities):
    members = list(comm)
    print(f"  Community {i}: {members}")

## 6. Generate Enriched Text Summaries for Each Entity

For every driver and rider, we generate a rich natural-language summary by walking their graph neighbourhood — this is the "text chunk" that gets embedded and stored in Qdrant.

In [ ]:
def get_zone_name(zone_id):
    z = next((z for z in zones if z["zone_id"] == zone_id), None)
    return z["name"] if z else zone_id

def get_review_texts_for_driver(driver_id):
    return [
        rev["text"] for rev in reviews
        if rev["reviewee_id"] == driver_id and rev["reviewer_type"] == "rider"
    ]

def driver_to_text(d):
    zone_name = get_zone_name(d["zone"])
    driver_trips = [t for t in trips if t["driver_id"] == d["driver_id"]]
    review_texts = get_review_texts_for_driver(d["driver_id"])
    riders_served = list({t["rider_id"] for t in driver_trips})
    rider_names = [r["name"] for r in riders if r["rider_id"] in riders_served]
    community_id = node_to_community.get(d["driver_id"], -1)
    
    badge_str = f" — badge: {d['badge']}" if d["badge"] else ""
    langs = ", ".join(d["languages"])
    trip_fares = [t["fare"] for t in driver_trips]
    avg_fare = sum(trip_fares) / len(trip_fares) if trip_fares else 0
    reviews_str = " | ".join(f'"{r}"' for r in review_texts) if review_texts else "No reviews yet."
    
    return (
        f"Driver profile: {d['name']} (ID: {d['driver_id']}) operates in {d['city']}, "
        f"primarily in the {zone_name} zone{badge_str}. "
        f"Vehicle: {d['vehicle']} ({d['vehicle_type']}). "
        f"Overall rating: {d['rating']}/5.0 across {d['trips_completed']} total trips. "
        f"Languages spoken: {langs}. "
        f"In our dataset, completed {len(driver_trips)} trips serving riders: {', '.join(rider_names) or 'none'}. "
        f"Average fare in dataset: ${avg_fare:.2f}. "
        f"Graph community: {community_id}. "
        f"Rider reviews: {reviews_str}"
    )

def rider_to_text(r):
    zone_name = get_zone_name(r["frequent_zone"])
    rider_trips = [t for t in trips if t["rider_id"] == r["rider_id"]]
    drivers_used = list({t["driver_id"] for t in rider_trips})
    driver_names = [d["name"] for d in drivers if d["driver_id"] in drivers_used]
    community_id = node_to_community.get(r["rider_id"], -1)
    rider_reviews = [rev for rev in reviews if rev["reviewer_id"] == r["rider_id"] and rev["reviewer_type"] == "rider"]
    
    return (
        f"Rider profile: {r['name']} (ID: {r['rider_id']}) based in {r['city']}, "
        f"frequently rides in {zone_name}. "
        f"Rider rating: {r['rating']}/5.0 across {r['total_trips']} total trips. "
        f"Preferred vehicle type: {r['preferred_type']}. "
        f"In our dataset, took {len(rider_trips)} trips with drivers: {', '.join(driver_names) or 'none'}. "
        f"Graph community: {community_id}. "
        f"Reviews left by this rider: {len(rider_reviews)} reviews."
    )

def zone_to_text(z):
    zone_trips = [t for t in trips if t["pickup_zone"] == z["zone_id"] or t["dropoff_zone"] == z["zone_id"]]
    zone_drivers = [d for d in drivers if d["zone"] == z["zone_id"]]
    surge_str = "Surge pricing is currently active." if z["surge_active"] else "No surge pricing."
    community_id = node_to_community.get(z["zone_id"], -1)
    
    return (
        f"Zone profile: {z['name']} (ID: {z['zone_id']}) in {z['city']}. "
        f"Demand level: {z['demand_level']}. Average wait: {z['avg_wait_min']} minutes. {surge_str} "
        f"Active drivers in this zone: {', '.join(d['name'] for d in zone_drivers) or 'none'}. "
        f"Trips involving this zone: {len(zone_trips)}. "
        f"Graph community: {community_id}."
    )

# Build document list for all entities
documents = []
for d in drivers:
    documents.append({
        "id": d["driver_id"], "entity_type": "driver",
        "name": d["name"], "text": driver_to_text(d),
        "metadata": d, "community": node_to_community.get(d["driver_id"], -1)
    })
for r in riders:
    documents.append({
        "id": r["rider_id"], "entity_type": "rider",
        "name": r["name"], "text": rider_to_text(r),
        "metadata": r, "community": node_to_community.get(r["rider_id"], -1)
    })
for z in zones:
    documents.append({
        "id": z["zone_id"], "entity_type": "zone",
        "name": z["name"], "text": zone_to_text(z),
        "metadata": z, "community": node_to_community.get(z["zone_id"], -1)
    })

print(f"Generated {len(documents)} enriched document summaries\n")
print("Sample document (D001):")
print("-" * 70)
print(next(doc["text"] for doc in documents if doc["id"] == "D001"))

## 7. Generate Embeddings (Dense) with OpenAI

In [ ]:
import numpy as np
from tqdm import tqdm

def embed_texts(texts: list[str], batch_size: int = 20) -> list[list[float]]:
    """Embed texts using OpenAI in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embeddings.extend([item.embedding for item in response.data])
    return all_embeddings

texts = [doc["text"] for doc in documents]
dense_embeddings = embed_texts(texts)

print(f"Generated {len(dense_embeddings)} dense vectors of dimension {len(dense_embeddings[0])}")

## 8. Build Sparse Vectors with BM25

Qdrant supports sparse vectors natively. We use BM25 to produce term-weight sparse representations — this enables keyword/lexical search alongside semantic search.

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"\b[a-z0-9]+\b", text.lower())

tokenized_corpus = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

# Build vocabulary → index mapping
vocab = {}
for tokens in tokenized_corpus:
    for tok in tokens:
        if tok not in vocab:
            vocab[tok] = len(vocab)

def rebuild_bm25(new_text: str) -> None:
    """Extend the BM25 model and vocab with tokens from a newly ingested document.

    Call this whenever a new entity is added so that its unique terms become
    searchable via sparse vectors in subsequent queries.
    """
    global bm25, tokenized_corpus, vocab
    new_tokens = tokenize(new_text)
    tokenized_corpus.append(new_tokens)
    bm25 = BM25Okapi(tokenized_corpus)
    for tok in new_tokens:
        if tok not in vocab:
            vocab[tok] = len(vocab)

def text_to_sparse_vector(text: str) -> tuple[list[int], list[float]]:
    """Convert text to a sparse vector (indices, values) using BM25 term weights."""
    tokens = tokenize(text)
    doc_tokens = set(tokens)
    indices, values = [], []
    for tok in doc_tokens:
        if tok in vocab:
            idx = vocab[tok]
            weight = float(bm25.idf.get(tok, 0.0))
            tf = tokens.count(tok)
            k1, b, avg_dl = 1.5, 0.75, bm25.avgdl
            dl = len(tokens)
            numerator = weight * tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / avg_dl)
            w = numerator / denominator if denominator > 0 else 0.0
            if w > 0:
                indices.append(idx)
                values.append(w)
    return indices, values

sparse_vectors = [text_to_sparse_vector(t) for t in texts]

# Print sample
sample_idx, sample_val = sparse_vectors[0]
top_terms = sorted(zip(sample_idx, sample_val), key=lambda x: -x[1])[:8]
rev_vocab = {v: k for k, v in vocab.items()}
print(f"Vocabulary size: {len(vocab)} terms")
print(f"Sample sparse vector for D001 — top BM25 terms:")
for idx, weight in top_terms:
    print(f"  '{rev_vocab[idx]}': {weight:.4f}")

## 9. Create Qdrant Collection

We create a collection with **two named vector spaces**:
- `dense`  — 1536-d OpenAI embeddings (cosine)
- `sparse` — BM25 sparse vectors

In [ ]:
from qdrant_client.models import (
    VectorsConfig, SparseVectorsConfig,
    VectorParams, SparseVectorParams, SparseIndexParams, Distance
)

# Drop collection if it already exists (clean slate for the PoC)
existing = [c.name for c in qdrant_client.get_collections().collections]
if COLLECTION_NAME in existing:
    qdrant_client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)

print(f"Collection '{COLLECTION_NAME}' created successfully.")
print(f"  Dense vector: {EMBED_DIM}d cosine")
print(f"  Sparse vector: BM25 (on-memory index)")

## 10. CRUD Operations on Qdrant

### 10.1 CREATE — Upsert all entity documents

In [ ]:
from qdrant_client.models import PointStruct, SparseVector

def make_point_id(entity_id: str) -> int:
    """Convert string IDs like 'D001', 'R002', 'Z003' to stable integers."""
    prefix_map = {"D": 1000, "R": 2000, "Z": 3000, "T": 4000}
    prefix = entity_id[0]
    number = int(entity_id[1:])
    return prefix_map.get(prefix, 9000) + number

points = []
for doc, dense_vec, (sp_indices, sp_values) in zip(documents, dense_embeddings, sparse_vectors):
    point_id = make_point_id(doc["id"])
    payload = {
        "entity_id":   doc["id"],
        "entity_type": doc["entity_type"],
        "name":        doc["name"],
        "text":        doc["text"],
        "community":   doc["community"],
        **{
            # Flatten metadata into payload for filtering
            k: v for k, v in doc["metadata"].items()
            if not isinstance(v, list)   # Qdrant handles scalars directly
        },
        "languages":   doc["metadata"].get("languages", []),
    }

    points.append(PointStruct(
        id=point_id,
        vector={
            "dense": dense_vec,
            "sparse": SparseVector(indices=sp_indices, values=sp_values)
        },
        payload=payload
    ))

result = qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Upserted {len(points)} points → status: {result.status}")
for p in points:
    print(f"  id={p.id:5d}  entity_type={p.payload['entity_type']:6s}  name={p.payload['name']}")

### 10.2 READ — Retrieve a specific point by ID

In [ ]:
# Fetch Carlos Rivera (D001) by point ID
target_id = make_point_id("D001")

results = qdrant_client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[target_id],
    with_payload=True,
    with_vectors=False  # skip vector data for readability
)

if results:
    record = results[0]
    print(f"READ result for point {target_id}:")
    print(f"  entity_id  : {record.payload['entity_id']}")
    print(f"  name       : {record.payload['name']}")
    print(f"  entity_type: {record.payload['entity_type']}")
    print(f"  rating     : {record.payload.get('rating')}")
    print(f"  community  : {record.payload.get('community')}")
    print(f"  text (snippet): {record.payload['text'][:120]}...")
else:
    print("No record found.")

### 10.3 UPDATE — Modify a driver's payload (e.g. rating update from new trips)

In [ ]:
from qdrant_client.models import SetPayload

# Simulate: Carlos Rivera completes more trips and earns an even higher rating
updated_fields = {
    "rating":           4.96,
    "trips_completed":  2400,   # +59 since last sync
    "badge":            "top_rated_platinum"  # new badge tier
}

# Step 1: update scalar payload fields (fast, no re-embedding needed for filter-only fields)
result = qdrant_client.set_payload(
    collection_name=COLLECTION_NAME,
    payload=updated_fields,
    points=[make_point_id("D001")]
)
print(f"Payload UPDATE status: {result.status}")

# Step 2: regenerate the text summary and vectors so search context reflects the new values.
# Any field that appears in the embedded text must trigger a full re-upsert.
d001_record = next(d for d in drivers if d["driver_id"] == "D001")
d001_record.update(updated_fields)   # keep in-memory source of truth consistent

updated_text = driver_to_text(d001_record)
updated_dense = embed_texts([updated_text])[0]
sp_i, sp_v    = text_to_sparse_vector(updated_text)

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=[PointStruct(
        id=make_point_id("D001"),
        vector={
            "dense":  updated_dense,
            "sparse": SparseVector(indices=sp_i, values=sp_v)
        },
        payload={
            **updated_fields,
            "text": updated_text,
        }
    )]
)
print("Vectors + text re-upserted with updated profile.")

# Verify
updated_rec = qdrant_client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[make_point_id("D001")],
    with_payload=True, with_vectors=False
)[0]
print(f"\nPost-update values for {updated_rec.payload['name']}:")
print(f"  rating          : {updated_rec.payload['rating']}")
print(f"  trips_completed : {updated_rec.payload['trips_completed']}")
print(f"  badge           : {updated_rec.payload['badge']}")
print(f"  text (snippet)  : {updated_rec.payload['text'][:120]}...")

### 10.4 DELETE — Remove a specific entity from the index

In [ ]:
from qdrant_client.models import PointIdsList

# Add a temporary "stale" driver to demonstrate deletion
stale_id = 9999
stale_text = "Driver profile: Test Driver (ID: D999) — temporary entry for deletion demo."
stale_emb  = embed_texts([stale_text])[0]
sp_i, sp_v = text_to_sparse_vector(stale_text)

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=[PointStruct(
        id=stale_id,
        vector={"dense": stale_emb, "sparse": SparseVector(indices=sp_i, values=sp_v)},
        payload={"entity_id": "D999", "entity_type": "driver", "name": "Test Driver",
                 "text": stale_text}
    )]
)
print(f"Inserted temporary point {stale_id}")

count_before = qdrant_client.count(collection_name=COLLECTION_NAME).count
print(f"Collection size before delete: {count_before}")

# DELETE
del_result = qdrant_client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=PointIdsList(points=[stale_id])
)
count_after = qdrant_client.count(collection_name=COLLECTION_NAME).count
print(f"DELETE status: {del_result.status}")
print(f"Collection size after delete:  {count_after}")

## 11. Search Operations

### 11.1 Dense Vector Search (Semantic)

In [ ]:
def dense_search(query: str, top_k: int = 5, entity_type_filter: str = None):
    """Semantic search using OpenAI dense embeddings."""
    query_vec = embed_texts([query])[0]

    search_filter = None
    if entity_type_filter:
        search_filter = Filter(
            must=[FieldCondition(key="entity_type", match=MatchValue(value=entity_type_filter))]
        )

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        using="dense",
        limit=top_k,
        query_filter=search_filter,
        with_payload=True
    )
    return results.points

# Query 1: Who are the best electric vehicle drivers?
q1 = "top rated electric vehicle driver with many completed trips"
print(f"Dense search: '{q1}'")
print("=" * 60)
for r in dense_search(q1, top_k=4):
    print(f"  [{r.score:.4f}] {r.payload['name']} ({r.payload['entity_type']}) "
          f"— rating: {r.payload.get('rating', 'N/A')}")

print()

# Query 2: High demand zones with surge pricing
q2 = "high demand zone with surge pricing and short wait times"
print(f"Dense search: '{q2}'")
print("=" * 60)
for r in dense_search(q2, top_k=3, entity_type_filter="zone"):
    print(f"  [{r.score:.4f}] {r.payload['name']} — demand: {r.payload.get('demand_level')} "
          f"surge: {r.payload.get('surge_active')}")

### 11.2 Sparse Vector Search (Keyword / BM25)

In [ ]:
def sparse_search(query: str, top_k: int = 5, entity_type_filter: str = None):
    """BM25-based keyword search using sparse vectors."""
    sp_indices, sp_values = text_to_sparse_vector(query)

    search_filter = None
    if entity_type_filter:
        search_filter = Filter(
            must=[FieldCondition(key="entity_type", match=MatchValue(value=entity_type_filter))]
        )

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=SparseVector(indices=sp_indices, values=sp_values),
        using="sparse",
        limit=top_k,
        query_filter=search_filter,
        with_payload=True
    )
    return results.points

# Keyword search for specific terms
q3 = "luxury BMW San Jose platinum"
print(f"Sparse (BM25) search: '{q3}'")
print("=" * 60)
for r in sparse_search(q3, top_k=4):
    print(f"  [{r.score:.4f}] {r.payload['name']} ({r.payload['entity_type']}) "
          f"— {r.payload.get('vehicle', r.payload.get('city', ''))}")

print()

q4 = "Tesla electric Mission District"
print(f"Sparse (BM25) search: '{q4}'")
print("=" * 60)
for r in sparse_search(q4, top_k=4):
    print(f"  [{r.score:.4f}] {r.payload['name']} ({r.payload['entity_type']})")

### 11.3 Hybrid Search (Dense + Sparse + RRF Fusion)

Qdrant's **Reciprocal Rank Fusion (RRF)** combines results from both vector spaces — best of semantic understanding AND keyword precision.

In [ ]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion, Query as QdrantQuery

def hybrid_search(query: str, top_k: int = 5, entity_type_filter: str = None):
    """Hybrid search combining dense (semantic) + sparse (BM25) via RRF."""
    # Dense query vector
    query_dense = embed_texts([query])[0]

    # Sparse query vector
    sp_indices, sp_values = text_to_sparse_vector(query)
    query_sparse = SparseVector(indices=sp_indices, values=sp_values)

    search_filter = None
    if entity_type_filter:
        search_filter = Filter(
            must=[FieldCondition(key="entity_type", match=MatchValue(value=entity_type_filter))]
        )

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=query_dense,  using="dense",  limit=top_k * 2),
            Prefetch(query=query_sparse, using="sparse", limit=top_k * 2),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        query_filter=search_filter,
        with_payload=True
    )
    return results.points

# Hybrid query 1: Best drivers for eco-conscious riders
q5 = "eco-friendly Tesla electric driver high rating professional"
print(f"Hybrid search: '{q5}'")
print("=" * 60)
for r in hybrid_search(q5, top_k=5):
    print(f"  [{r.score:.4f}] {r.payload['name']} ({r.payload['entity_type']}) "
          f"— vehicle: {r.payload.get('vehicle', 'N/A')}, rating: {r.payload.get('rating', 'N/A')}")

print()

# Hybrid query 2: Cross-entity — find drivers AND zones related to Silicon Valley
q6 = "Silicon Valley San Jose luxury high demand surge zone driver"
print(f"Hybrid search: '{q6}'")
print("=" * 60)
for r in hybrid_search(q6, top_k=5):
    etype = r.payload['entity_type']
    extra = r.payload.get('vehicle', r.payload.get('demand_level', ''))
    print(f"  [{r.score:.4f}] [{etype:6s}] {r.payload['name']} — {extra}")

### 11.4 Hybrid Search with Payload Filtering

Demonstrating Qdrant's pre-filter capability — filter by payload fields AND search vectors simultaneously.

In [ ]:
# Find high-performing drivers in San Francisco only
q7 = "reliable driver with great reviews and many trips"

sf_driver_filter = Filter(must=[
    FieldCondition(key="entity_type", match=MatchValue(value="driver")),
    FieldCondition(key="city",        match=MatchValue(value="San Francisco")),
])

query_dense  = embed_texts([q7])[0]
sp_i, sp_v   = text_to_sparse_vector(q7)

filtered_results = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense,                        using="dense",  limit=10),
        Prefetch(query=SparseVector(indices=sp_i, values=sp_v), using="sparse", limit=10),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
    query_filter=sf_driver_filter,
    with_payload=True
)

print(f"Hybrid search with filter (entity_type=driver, city=San Francisco):")
print(f"Query: '{q7}'")
print("=" * 60)
for r in filtered_results.points:
    print(f"  [{r.score:.4f}] {r.payload['name']} — rating: {r.payload.get('rating')} "
          f"trips: {r.payload.get('trips_completed')} badge: {r.payload.get('badge')}")

## 12. GraphRAG-Enhanced Answer Generation

Here we tie everything together: use hybrid search to retrieve the most relevant graph-enriched context, then pass it to GPT-4o to generate a grounded, knowledge-graph-aware answer.

In [ ]:
def graphrag_answer(question: str, top_k: int = 5, entity_type_filter: str = None) -> str:
    """
    Full GraphRAG + Qdrant pipeline:
      1. Hybrid search Qdrant for relevant graph-enriched documents
      2. Inject retrieved context into GPT-4o prompt
      3. Return grounded answer
    """
    retrieved = hybrid_search(question, top_k=top_k, entity_type_filter=entity_type_filter)

    if not retrieved:
        return "No relevant context found in the knowledge graph."

    # Build context block from retrieved graph summaries
    context_parts = []
    for i, point in enumerate(retrieved, 1):
        p = point.payload
        context_parts.append(
            f"[{i}] Entity: {p['name']} (type={p['entity_type']}, "
            f"community={p.get('community', '?')}, score={point.score:.3f})\n"
            f"    {p['text']}"
        )
    context_block = "\n\n".join(context_parts)

    system_prompt = (
        "You are a knowledgeable assistant for a ride-sharing marketplace platform. "
        "You have access to a knowledge graph of drivers, riders, trips, and zones. "
        "Use ONLY the provided context to answer questions. "
        "Cite specific entities and graph relationships in your answer. "
        "Be concise but informative."
    )

    user_prompt = (
        f"Context from Knowledge Graph (retrieved via hybrid search):\n\n"
        f"{context_block}\n\n"
        f"Question: {question}\n\n"
        f"Answer based on the knowledge graph context above:"
    )

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=600
    )

    return response.choices[0].message.content


# ── Demo Q&A ─────────────────────────────────────────────────────────────────
questions = [
    "Which drivers should I recommend to a rider who wants a premium electric vehicle experience?",
    "What can you tell me about the demand patterns across different zones? Are there surge pricing concerns?",
    "Which rider has the most experience with luxury rides and which driver did they primarily use?",
]

for q in questions:
    print(f"\n{'━'*70}")
    print(f"Q: {q}")
    print(f"{'━'*70}")
    answer = graphrag_answer(q)
    print(f"A: {answer}")

## 13. Programmatic CRUD from New Data (Live Ingestion Demo)

This section demonstrates how new records arriving from the relational database are ingested on-demand — the core value proposition for agentic applications.

In [ ]:
def ingest_new_driver(driver_record: dict):
    """Ingest a new driver record into the knowledge graph and Qdrant."""
    # 1. Add to knowledge graph
    G.add_node(driver_record["driver_id"], **driver_record, node_type="driver")
    print(f"  Graph: added node {driver_record['driver_id']}")

    # 2. Generate graph-enriched text summary
    text = (
        f"Driver profile: {driver_record['name']} (ID: {driver_record['driver_id']}) "
        f"operates in {driver_record['city']}, "
        f"vehicle: {driver_record['vehicle']} ({driver_record['vehicle_type']}). "
        f"Rating: {driver_record['rating']}/5.0 across {driver_record['trips_completed']} trips. "
        f"Languages: {', '.join(driver_record['languages'])}. "
        f"Badge: {driver_record.get('badge', 'none')}. Newly onboarded driver."
    )

    # 3. Extend the BM25 model with any new tokens introduced by this document.
    #    Without this step, unique terms like the driver's vehicle model or badge name
    #    would be silently dropped from the sparse vector.
    vocab_size_before = len(vocab)
    rebuild_bm25(text)
    new_terms = len(vocab) - vocab_size_before
    print(f"  BM25: vocab extended by {new_terms} new terms (total: {len(vocab)})")

    # 4. Generate embeddings (after BM25 rebuild so sparse uses updated index)
    dense_vec = embed_texts([text])[0]
    sp_i, sp_v = text_to_sparse_vector(text)

    # 5. Upsert to Qdrant
    point_id = make_point_id(driver_record["driver_id"])
    result = qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=[PointStruct(
            id=point_id,
            vector={
                "dense":  dense_vec,
                "sparse": SparseVector(indices=sp_i, values=sp_v)
            },
            payload={
                "entity_id":   driver_record["driver_id"],
                "entity_type": "driver",
                "name":        driver_record["name"],
                "text":        text,
                "community":   -1,  # recalculated on next graph refresh
                **{k: v for k, v in driver_record.items() if not isinstance(v, list)},
                "languages":   driver_record.get("languages", [])
            }
        )]
    )
    print(f"  Qdrant: upserted point {point_id} → status: {result.status}")
    return point_id


# Simulate a new driver arriving from the database
new_driver = {
    "driver_id": "D006",
    "name": "Sofia Reyes",
    "city": "San Francisco",
    "rating": 4.89,
    "trips_completed": 542,
    "vehicle": "Ford Mustang Mach-E",
    "vehicle_type": "electric",
    "badge": "rising_star",
    "languages": ["English", "Spanish", "Portuguese"],
    "zone": "Z002"
}

print(f"Ingesting new driver: {new_driver['name']}")
new_point_id = ingest_new_driver(new_driver)

print(f"\nTotal collection size: {qdrant_client.count(collection_name=COLLECTION_NAME).count}")

# Verify new driver is searchable — including by terms that were out-of-vocabulary before ingestion
print("\nSearching for the new driver via hybrid search (sparse can now match 'mustang', 'rising_star', 'portuguese'):")
results = hybrid_search("Mustang electric rising star Portuguese driver", top_k=3)
for r in results:
    print(f"  [{r.score:.4f}] {r.payload['name']} ({r.payload.get('badge', 'no badge')})")

## 14. Search Comparison: Dense vs Sparse vs Hybrid

Side-by-side comparison to show why hybrid wins.

In [ ]:
import pandas as pd

def compare_search_methods(query: str, top_k: int = 5):
    """Run all three search methods and display a comparison table."""
    dense_res  = dense_search(query,  top_k=top_k)
    sparse_res = sparse_search(query, top_k=top_k)
    hybrid_res = hybrid_search(query, top_k=top_k)

    def fmt(results):
        return [f"{r.payload['name']} ({r.payload['entity_type'][:3]}) [{r.score:.3f}]"
                for r in results]

    dense_list  = fmt(dense_res)
    sparse_list = fmt(sparse_res)
    hybrid_list = fmt(hybrid_res)

    max_len = max(len(dense_list), len(sparse_list), len(hybrid_list))
    while len(dense_list)  < max_len: dense_list.append("")
    while len(sparse_list) < max_len: sparse_list.append("")
    while len(hybrid_list) < max_len: hybrid_list.append("")

    df = pd.DataFrame({
        "Rank":   list(range(1, max_len + 1)),
        "Dense (Semantic)": dense_list,
        "Sparse (BM25)":    sparse_list,
        "Hybrid (RRF)":     hybrid_list,
    })
    return df

test_query = "top rated driver electric vehicle San Francisco professional reviews"
print(f"Search comparison for: '{test_query}'\n")
df_compare = compare_search_methods(test_query)
print(df_compare.to_string(index=False))
print()
print("Note: Hybrid (RRF) blends semantic relevance with keyword precision,")
print("resulting in more robust ranking across both query types.")

## 15. Summary

### What we built

| Component | Technology | Purpose |
|-----------|-----------|----------|
| Source data | JSON arrays (mock DB) | Drivers, riders, trips, reviews, zones |
| Knowledge graph | NetworkX DiGraph | Entity + relationship modelling |
| Community detection | Greedy modularity | GraphRAG-style clustering |
| Text summaries | Python (graph traversal) | Rich per-entity context for embedding |
| Dense embeddings | OpenAI text-embedding-3-small | Semantic search |
| Sparse vectors | BM25 (rank-bm25) | Keyword / lexical search |
| Vector store | Qdrant Serverless | CRUD + hybrid search |
| Hybrid fusion | Qdrant RRF | Best-of-both ranking |
| Answer generation | OpenAI GPT-4o | Grounded RAG responses |

### Key patterns for production

1. **On-demand ingestion** — `ingest_new_driver()` pattern scales to any entity type; call it from a DB change-data-capture (CDC) stream
2. **Graph refresh** — periodically recompute communities and regenerate summaries for entities whose neighbourhood changed
3. **Filtered hybrid search** — Qdrant payload filters run BEFORE vector search, so they're O(filtered) not O(all)
4. **Entity types as namespaces** — the `entity_type` filter lets you scope searches to drivers, riders, or zones depending on the agent's current task
5. **Qdrant named vectors** — a single collection handles both vector spaces; no dual-collection fan-out needed